# §F — Served LLM+NTK NLL gate (end-to-end)

Release-gate check for **NTK Phase-2**: confirm the *served* `ntk_model` ISVC reproduces the controller's hook-attached reference NLL.

Flow (each step optional if already done):
1. *(optional)* **register** an existing `controller.pt` as an `ntk_controller` row (skip if you fine-tuned via the pipeline, which already registers it).
2. **deploy** the `ntk_model` ISVC via CogAPI `POST /models-serving`, and wait until it's ready.
3. **resolve** the endpoint + served model name via cogflow (no manual URL).
4. **measure** served NLL over `/v1/completions` (echo + logprobs) and compare to the reference.

Same metric as §E `evaluate_nll.py` `_completion_nll` (teacher-forced, completion-masked). Apples-to-apples by sending the **exact token ids** as the completions prompt. **Gate:** `|served − reference| / reference ≤ threshold` (default 1%).

Prereqs: cogflow installed + Kubernetes access (your namespace); a base LLM `model_info` row; the ISVC deployed **without** `--quantization`; `REFERENCE_NLL` computed (§E reference arm) on the **same** controller.


In [ ]:
# !pip install transformers requests cogflow

In [ ]:
# --- Config -------------------------------------------------------------
BASE = "Qwen/Qwen2.5-0.5B-Instruct"          # HF id of the base the ISVC serves
EVAL_DATASET_ID = "39030a5d-b36c-4f07-8895-667515fcaa14"  # cogflow dataset: GSM8K eval split
#   Held-out GSM8K *test* split (32 rows), disjoint from the fine-tune train split
#   (a6693b90). Registered as a platform dataset so this notebook is self-contained:
#   it pulls eval data through cogflow, the same channel the fine-tune uses for train.
EVAL_JSONL = None                            # resolved below by the cogflow download cell

# Identity of the rows + the ISVC. NAMESPACE/endpoint/served-name auto-resolve
# from your user context; you normally set only these:
BASE_MODEL_ID = "8f7318ab-6e4e-4e94-aeb9-6a61b6dd51e6"  # dev: Qwen2.5-0.5B-Instruct llm row (GET /models?type=llm)
NTK_CONTROLLER_ID = "1a62543f-52f7-4871-82a7-f25a6dcd33f5"  # dev: validated ntk_controller (Qwen2.5-0.5B GSM8K)
ISVC_NAME = "ntk-gsm8k"                            # name to deploy/serve under

# (optional) reuse an existing controller.pt instead of fine-tuning: set the
# path and the next cell registers it as an ntk_controller (sets NTK_CONTROLLER_ID).
CONTROLLER_PT = None                               # e.g. "controller.pt"; None = skip registration
DATASET_ID = "a6693b90-b11a-4af1-a31b-51cd3c1a84e8"   # JSONL GSM8K dataset for the fine-tune option
RUN_FINETUNE = False                               # True -> submit an ntk_model fine-tune to create the controller

NAMESPACE = None                              # None -> your user namespace (cogflow common.get_namespace())
COGAPI = None                                 # None -> cogflow config.API_PATH; else CogAPI base URL
USER = None                                   # None -> cogflow common.get_current_user(); else your kubeflow-userid
ENDPOINT = None                               # None -> resolve via cogflow; else override
SERVED_MODEL_NAME = None                      # None -> read from /v1/models; else override

API_KEY = None                                # only for external-ingress/cross-ns access; in-namespace needs none
REFERENCE_NLL = 0.5886                        # §E reference arm on the SERVED controller 1a62543f
#   Computed on this eval set (39030a5d) with the serving ntkmirror 0.4.0:
#   floor (base only) = 0.7706 ; reference (hook-attached) = 0.5886 ; controller.pt
#   sha256 52d9ea006a71792ba381145bbb6b0a3fb0811a85e2b6a6712e662c41355f5e1f.
#   Re-derive if you serve a different controller (see README runbook §1).
THRESHOLD = 0.01                              # |served-reference|/reference pass band
MAX_TOKENS = 0                                # 0 = echo only; auto-falls back to 1 if rejected

## 1. (optional) Register an existing `controller.pt`
Only if you have a `controller.pt` and did **not** fine-tune via the pipeline (which already registers the row). Sets `NTK_CONTROLLER_ID`. No training — just logs the artifact + the catalog handshake.


In [ ]:
if CONTROLLER_PT:
    import cogflow.core.models as cogflow_models
    with cogflow_models.start_run(run_name=f"ntk-reuse-{ISVC_NAME}") as run:
        run_id = run.info.run_id
        cogflow_models.log_artifact(CONTROLLER_PT, artifact_path="controller")  # -> controller/controller.pt
    NTK_CONTROLLER_ID = cogflow_models.register_finetuned_catalog_entry(
        run_id=run_id,
        served_model_name=ISVC_NAME,
        adapter_type="ntk_controller",
        base_model_id=BASE_MODEL_ID,
        base_model_hf_id=BASE,
        user_id=USER,  # None -> common.get_current_user()
    )
    print("registered ntk_controller row:", NTK_CONTROLLER_ID)
else:
    print("CONTROLLER_PT not set -> using existing NTK_CONTROLLER_ID:", NTK_CONTROLLER_ID)

## 1b. (alternative) Fine-tune to create the `ntk_controller`

Use this instead of §1 when you have no `controller.pt`: submit an `ntk_model` fine-tune over `BASE_MODEL_ID` + `DATASET_ID`, then poll until the run registers the `ntk_controller` row (its id becomes `NTK_CONTROLLER_ID`). Needs GPU/kfp; can take a while. Set `RUN_FINETUNE = True` to enable.


In [ ]:
import time
import requests
from cogflow import serving as cogflow_serving

# Lazy-resolve CogAPI base + user (same as the deploy cell; idempotent).
if COGAPI is None:
    from cogflow.config import config as _cog_config
    COGAPI = _cog_config.API_PATH
if USER is None:
    from cogflow.utils import common as _cog_common
    USER = _cog_common.get_current_user()


def list_ntk_controllers():
    """ntk_controller rows for BASE_MODEL_ID (empty list when none -> 404)."""
    r = requests.get(
        f"{COGAPI.rstrip('/')}/models",
        params={"type": "ntk_controller", "base_model_id": BASE_MODEL_ID},
        headers={"kubeflow-userid": USER}, timeout=30,
    )
    if r.status_code == 404:
        return []
    r.raise_for_status()
    return [m["id"] for m in r.json().get("data", [])]


if RUN_FINETUNE:
    before = set(list_ntk_controllers())
    body = {
        "base_model_id": BASE_MODEL_ID,
        "dataset_id": DATASET_ID,
        "output_name": ISVC_NAME,
        "method": "ntk",
        "export": "ntk_model",
        "hyperparams": {"gates": 5000, "max_log_gate": 0.05, "train_steps": 240, "lr": 5e-3},
    }
    r = requests.post(
        f"{COGAPI.rstrip('/')}/models/fine-tune", json=body,
        headers={"Content-Type": "application/json", "kubeflow-userid": USER}, timeout=120,
    )
    if r.status_code >= 400:
        raise RuntimeError(f"fine-tune failed [{r.status_code}]: {r.text}")
    resp = r.json().get("data", {})
    print("fine-tune submitted:", resp)  # {model_id (reserved correlation id), run_id (kfp run)}

    # The kfp run trains + registers the ntk_controller at completion (the row
    # id is the MLflow run id, not the returned ids) — poll for the new row.
    deadline = time.time() + 3600  # 1h
    while True:
        new = set(list_ntk_controllers()) - before
        if new:
            NTK_CONTROLLER_ID = sorted(new)[-1]
            print("ntk_controller registered:", NTK_CONTROLLER_ID)
            break
        if time.time() > deadline:
            raise TimeoutError("fine-tune did not register an ntk_controller within timeout")
        print("waiting for fine-tune to register the controller ...")
        time.sleep(30)
else:
    print("RUN_FINETUNE=False -> skipping fine-tune; using NTK_CONTROLLER_ID:", NTK_CONTROLLER_ID)

## 2. Deploy the `ntk_model` ISVC (via CogAPI) and wait for ready
`POST /models-serving` with `model_id` = base LLM row + `llm_adapter:{kind:'ntk_model', adapter_model_id:<row>}`. CogAPI resolves the controller artifact, sets `--hf-overrides` + `storageUri`, and serves on the NTK-enabled runtime. Idempotent: skips the create if the ISVC already exists.


In [ ]:
import time
import requests
from cogflow import serving as cogflow_serving

# Resolve CogAPI base + user from cogflow if not set explicitly.
if COGAPI is None:
    from cogflow.config import config as _cog_config
    COGAPI = _cog_config.API_PATH
if USER is None:
    from cogflow.utils import common as _cog_common
    USER = _cog_common.get_current_user()

def isvc_status(isvc_name, namespace=None):
    models = cogflow_serving.list_models(namespace=namespace, isvc_name=isvc_name)
    return (models[0].get("status") if models else None)

existing = isvc_status(ISVC_NAME, NAMESPACE)
if existing is not None:
    print(f"ISVC {ISVC_NAME!r} already exists (status={existing!r}); skipping create")
else:
    body = {
        "isvc_name": ISVC_NAME,
        "model_id": BASE_MODEL_ID,
        "llm_adapter": {"kind": "ntk_model", "adapter_model_id": NTK_CONTROLLER_ID},
    }
    headers = {"Content-Type": "application/json", "kubeflow-userid": USER}
    r = requests.post(f"{COGAPI.rstrip('/')}/models-serving", json=body, headers=headers, timeout=120)
    if r.status_code >= 400:
        raise RuntimeError(f"deploy failed [{r.status_code}]: {r.text}")  # e.g. 422 if cogflow gate not satisfied
    print("deploy accepted:", r.json())

# Wait for the predictor to become ready.
deadline = time.time() + 900  # 15 min
while True:
    st = isvc_status(ISVC_NAME, NAMESPACE)
    print("status:", st)
    if st == "ready":
        break
    if time.time() > deadline:
        raise TimeoutError(f"ISVC {ISVC_NAME!r} not ready within timeout (last status={st!r})")
    time.sleep(15)

## 3. Resolve endpoint + served model name (via cogflow)


In [ ]:
def resolve_endpoint(isvc_name, namespace=None):
    """Look up the ISVC via cogflow and return the OpenAI base URL."""
    models = cogflow_serving.list_models(namespace=namespace, isvc_name=isvc_name)
    if not models:
        raise RuntimeError(f"ISVC {isvc_name!r} not found in namespace {namespace!r}")
    url = models[0].get("served_model_url")
    if not url:
        raise RuntimeError(f"ISVC {isvc_name!r} has no served_model_url (status={models[0].get('status')!r})")
    return url.rstrip("/") + "/openai/v1"   # KServe HF runtime mounts OpenAI here

def discover_served_model_name(endpoint, api_key=None):
    headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
    r = requests.get(endpoint.rstrip("/") + "/models", headers=headers, timeout=30)
    r.raise_for_status()
    data = r.json().get("data", [])
    if not data:
        raise RuntimeError("served /v1/models returned no models")
    return data[0]["id"]

if ENDPOINT is None:
    ENDPOINT = resolve_endpoint(ISVC_NAME, NAMESPACE)
    print("resolved endpoint :", ENDPOINT)
if SERVED_MODEL_NAME is None:
    SERVED_MODEL_NAME = discover_served_model_name(ENDPOINT, API_KEY)
print("served model name :", SERVED_MODEL_NAME)

## 3b. Download the eval set via cogflow
Pull the eval JSONL from the platform dataset channel (same path the fine-tune component uses: `download_dataset` → unwrap ZIP → find the `.jsonl`), so the benchmark depends on a registered cogflow dataset rather than a local file.


In [ ]:
import zipfile
from pathlib import Path
from cogflow import datasets as cogflow_datasets

def download_eval_jsonl(dataset_id, work_dir="runs/_eval"):
    """Download a cogflow dataset and return the local .jsonl path.

    CogAPI's download wraps the object in a ZIP (even single-file), so we
    unwrap and find exactly one .jsonl inside — mirroring the fine-tune
    component's dataset handling."""
    work = Path(work_dir); work.mkdir(parents=True, exist_ok=True)
    downloaded = Path(cogflow_datasets.download_dataset(dataset_id, output_path=str(work)))
    if not zipfile.is_zipfile(downloaded):
        return str(downloaded)  # already a raw jsonl
    extracted = work / "extracted"; extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(downloaded) as zf:
        zf.extractall(extracted)
    jsonls = sorted(p for p in extracted.rglob("*") if p.is_file() and p.suffix.lower() == ".jsonl")
    if len(jsonls) != 1:
        raise RuntimeError(f"expected exactly one .jsonl in dataset {dataset_id}, found {jsonls}")
    return str(jsonls[0])

if EVAL_JSONL is None:
    EVAL_JSONL = download_eval_jsonl(EVAL_DATASET_ID)
print("eval jsonl:", EVAL_JSONL)

## 4. Measure served NLL + compare to reference


In [ ]:
import json

def load_examples(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def completion_token_split(tokenizer, prompt, completion):
    """Same split as _completion_nll: prompt with special tokens, completion
    with add_special_tokens=False, concatenated. Returns int lists."""
    prompt_ids = tokenizer(prompt).input_ids
    completion_ids = tokenizer(completion, add_special_tokens=False).input_ids
    return prompt_ids, completion_ids, prompt_ids + completion_ids

def completion_logprobs_from_echo(logprobs_obj, input_ids, completion_start):
    if not isinstance(logprobs_obj, dict):
        raise RuntimeError(
            f"served response 'logprobs' is missing or not an object "
            f"(got {type(logprobs_obj).__name__}); the server returned no logprobs"
        )
    token_logprobs = logprobs_obj.get("token_logprobs")
    if token_logprobs is None:
        raise RuntimeError("served response has no logprobs.token_logprobs")
    if len(token_logprobs) < len(input_ids):
        raise RuntimeError(
            f"served logprobs length {len(token_logprobs)} < input length "
            f"{len(input_ids)}; cannot align completion span"
        )
    comp = token_logprobs[completion_start:len(input_ids)]
    if any(lp is None for lp in comp):
        raise RuntimeError(
            "served logprobs contain None inside the completion span; "
            "the server did not return a logprob for every completion token"
        )
    return comp

def _post_completion(url, headers, base_body, max_tokens):
    return requests.post(url, headers=headers, json={**base_body, "max_tokens": max_tokens}, timeout=120)

def served_completion_nll(endpoint, served_model_name, api_key, tokenizer, examples, max_tokens=0):
    """Teacher-forced completion-NLL via the served endpoint. Global sum of
    per-completion-token negative logprob / total completion tokens (matches
    _completion_nll). Falls back max_tokens 0 -> 1 if the server rejects 0."""
    url = endpoint.rstrip("/") + "/completions"
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    eff = max_tokens
    total_neg_logprob, total_tokens = 0.0, 0
    for i, ex in enumerate(examples):
        prompt_ids, completion_ids, input_ids = completion_token_split(
            tokenizer, ex["prompt"], ex["completion"])
        if not completion_ids:
            continue
        base_body = {"model": served_model_name, "prompt": input_ids,
                     "echo": True, "logprobs": 1, "temperature": 0}
        resp = _post_completion(url, headers, base_body, eff)
        if resp.status_code == 400 and eff == 0:
            eff = 1
            resp = _post_completion(url, headers, base_body, eff)
        resp.raise_for_status()
        choice = resp.json()["choices"][0]
        comp = completion_logprobs_from_echo(choice["logprobs"], input_ids, len(prompt_ids))
        total_neg_logprob += -float(sum(comp))
        total_tokens += len(comp)
        if (i + 1) % 10 == 0:
            print(f"[served] {i + 1} examples scored")
    if total_tokens == 0:
        raise RuntimeError("no completion tokens scored")
    return total_neg_logprob / total_tokens

In [ ]:
import math
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE)
examples = load_examples(EVAL_JSONL)
print(f"scoring {len(examples)} examples against {ENDPOINT} (model={SERVED_MODEL_NAME})")

served_nll = served_completion_nll(
    ENDPOINT, SERVED_MODEL_NAME, API_KEY, tokenizer, examples, max_tokens=MAX_TOKENS)

print("\n================== §F served-arm summary ==================")
print(f"served (LLM+NTK ISVC)       : {served_nll:.4f}  ({math.exp(served_nll):.4f} ppl)")
rel = abs(served_nll - REFERENCE_NLL) / max(REFERENCE_NLL, 1e-9)
verdict = "PASS" if rel <= THRESHOLD else "FAIL"
print(f"reference (reused §E hooks)  : {REFERENCE_NLL:.4f}  "
      f"(Δ = {served_nll - REFERENCE_NLL:+.4f}, relative = {rel*100:.2f}%, "
      f"threshold {THRESHOLD*100:.0f}%, {verdict})")

## Notes

- **Auth:** an in-namespace run needs **no token** — the namespace's Istio AuthorizationPolicy trusts same-namespace traffic, and CogAPI scopes by your `kubeflow-userid`. `API_KEY` is only for external-ingress/cross-namespace access.
- **Deploy gate:** a `422` from the deploy step means the CogAPI pod's cogflow lacks the ntk serving kwargs — it must be on `>=3.0.0b8` (rebuild on `cogflow:latest-beta`).
- **Reference must match the served controller.** `REFERENCE_NLL` is only valid if computed (§E reference arm) on the exact `controller.pt` this ISVC serves.
- **No quantization on the gate ISVC** — served (bf16/fp16) vs reference (fp32) should differ only by float noise (the 1% band).
- If the cluster `served_model_url` isn't reachable from where you run the notebook, set `ENDPOINT` to the externally-reachable URL.
